In [12]:
from deepagents.backends import LocalShellBackend

root_dir= "/home/cooper/githubProjects/agent-deploy-kit/agents/data_agent"
backend = LocalShellBackend(root_dir=root_dir, env={"PATH": "/usr/bin:/bin"})

/tmp/ipykernel_10597/398732899.py:4: LangChainDeprecationWarning: `LocalShellBackend` `virtual_mode` default will change in deepagents==0.6.0; please specify `virtual_mode` explicitly. Note: `virtual_mode` is for virtual path semantics (e.g., `CompositeBackend` routing) and optional path-based guardrails; it does not provide sandboxing or process isolation. Security note: leaving `virtual_mode=False` allows absolute paths and `'..'` to bypass `root_dir`, and `LocalShellBackend` provides no sandboxing (`execute()` runs commands on the host; `virtual_mode` does not restrict shell execution). See https://reference.langchain.com/python/deepagents/ for usage guidelines.
  backend = LocalShellBackend(root_dir=root_dir, env={"PATH": "/usr/bin:/bin"})


In [13]:
import csv
import io

# Create sample sales data
data = [
    ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
]

# Convert to CSV bytes
text_buf = io.StringIO()
writer = csv.writer(text_buf)
writer.writerows(data)
csv_bytes = text_buf.getvalue().encode("utf-8")
text_buf.close()

# Upload to backend
backend.upload_files([(f"{root_dir}/data/sales_data.csv", csv_bytes)])

[FileUploadResponse(path='/home/cooper/githubProjects/agent-deploy-kit/agents/data_agent/data/sales_data.csv', error=None)]

In [14]:
from langchain.tools import tool



@tool(parse_docstring=True)
def send_message(text: str, file_path: str | None = None) -> str:
    """Send message, optionally including attachments such as images.

    Args:
        text: (str) text content of the message
        file_path: (str) file path of attachment in the filesystem.
    """
    if not file_path:
        print(f"{text}")
    else:
        fp = backend.download_files([file_path])
        print(f"{fp[0].content}")

    return "Message sent."

In [18]:
from utils.langchain_model import get_singleton_client
from langchain_core.utils.uuid import uuid7

from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent
from langchain.messages import HumanMessage


checkpointer = InMemorySaver()

agent = create_deep_agent(
    model=get_singleton_client(llm_provider="longcat"),
    tools=[send_message],
    backend=backend,
    checkpointer=checkpointer,
)

thread_id = str(uuid7())
config={"configurable": {"thread_id": thread_id}}


input_message = {
    "role": "user",
    "content": (
        "Call send_message with text='analysis report' and file_path='./data/sales_data.csv'."
    ),
}


# agent.invoke(
#     {'messages': [input_message]},
#     config=config)

for chunk in agent.stream(
        {'messages': [input_message]},
        config=config,
        stream_mode='updates'):
    print(chunk)


{'PatchToolCallsMiddleware.before_agent': None}
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 6653, 'total_tokens': 6683, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 6528, 'image_tokens': 0, 'video_tokens': 0, 'text_tokens': 0}, 'effectiveCachedTokens': 6528, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'output_tokens_details': None, 'cached_tokens': 0}, 'model_provider': 'openai', 'model_name': 'LongCat-2.0-Preview', 'system_fingerprint': None, 'id': 'f5eb29ba43a64f8da3cec985308ab97d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ee028-5d52-7a01-92e9-b8fa0beb71b7-0', tool_calls=[{'name': 'send_message', 'args': {'text': 'analysis report', 'file_path': './data/sales_data.csv'}, 'id': 'call_32f5ea14529a4337a28711cd', 'type': 'tool_call'}], invalid_tool_calls=[]